# Preprocessing


In [34]:
import json
import os

# =========================
# PATHS
# =========================

INPUT_JSON = r"C:\Users\User\Downloads\archive\data\annotations.json"
OUTPUT_JSON = r"C:\Users\User\Downloads\archive\data\annotations_merged.json"

# =========================
# CATEGORY MAPPING
# =========================

category_mapping = {

"Cigarette": "Cigarette",
"Unlabeled litter": "Unlabeled",

# Plastic
"Plastic film": "Plastic",
"Other plastic": "Plastic",
"Other plastic wrapper": "Plastic",
"Plastic bottle cap": "Plastic",
"Plastic straw": "Plastic",
"Polypropylene bag": "Plastic",
"Plastic glooves": "Plastic",
"Tupperware": "Plastic",

# Bottle
"Clear plastic bottle": "Bottle",

# Metal
"Drink can": "Metal",
"Metal lid": "Metal",
"Aerosol": "Metal",
"Aluminium blister pack": "Metal",

# Glass
"Broken glass": "Glass",
"Glass cup": "Glass",
"Glass jar": "Glass",

# Paper
"Pizza box": "Paper",
"Egg carton": "Paper",
"Magazine paper": "Paper",
"Wrapping paper": "Paper",
"Toilet tube": "Paper",
"Paper straw": "Paper",

# Foam
"Foam cup": "Foam",
"Foam food container": "Foam",

# Food
"Food waste": "Other",

# Rare → Other
"Carded blister pack": "Other",
"Battery": "Other",
"Other plastic cup": "Other",
"Shoe": "Other",
"Six pack rings": "Other",
"Spread tub": "Other",
"Squeezable tube": "Other",

# Plastic related
"Disposable plastic cup": "Plastic",
"Other plastic bottle": "Plastic",
"Other plastic container": "Plastic",
"Plastic lid": "Plastic",
"Plastic utensils": "Plastic",
"Single-use carrier bag": "Plastic",
"Garbage bag": "Plastic",
"Crisp packet": "Plastic",

# Foam / Styrofoam
"Styrofoam piece": "Foam",

# Paper / cardboard
"Corrugated carton": "Paper",
"Meal carton": "Paper",
"Other carton": "Paper",
"Paper bag": "Paper",
"Paper cup": "Paper",
"Normal paper": "Paper",
"Tissues": "Paper",
"Drink carton": "Paper",
"Plastified paper bag": "Paper",

# Metal
"Aluminium foil": "Metal",
"Food Can": "Metal",
"Metal bottle cap": "Metal",
"Pop tab": "Metal",
"Scrap metal": "Metal",

# Glass
"Glass bottle": "Glass",

# Other
"Rope & strings": "Other",
"Disposable food container": "Other"
}

# =========================
# LOAD DATASET
# =========================

print("Loading dataset...")

with open(INPUT_JSON) as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]
categories = data["categories"]

print("Total images:", len(images))
print("Total annotations:", len(annotations))
print("Original categories:", len(categories))


# =========================
# CREATE CATEGORY LOOKUP
# =========================

category_lookup = {}

for cat in categories:
    category_lookup[cat["id"]] = cat["name"]


# =========================
# CREATE NEW CATEGORY LIST
# =========================

unique_classes = set(category_mapping.values())
unique_classes.add("Other")

new_category_ids = {}
new_categories = []

next_id = 1

for name in sorted(unique_classes):
    new_category_ids[name] = next_id
    new_categories.append({
        "id": next_id,
        "name": name
    })
    next_id += 1


print("New categories:", len(new_categories))


# =========================
# UPDATE ANNOTATIONS
# =========================

print("Updating annotations...")

for ann in annotations:

    old_cat_id = ann["category_id"]
    old_name = category_lookup[old_cat_id]

    new_name = category_mapping.get(old_name, "Other")

    ann["category_id"] = new_category_ids[new_name]


# =========================
# UPDATE DATASET STRUCTURE
# =========================

data["categories"] = new_categories


# =========================
# SAVE NEW DATASET
# =========================

print("Saving merged dataset...")

with open(OUTPUT_JSON, "w") as f:
    json.dump(data, f)

print("Done!")
print("Saved to:", OUTPUT_JSON)

Loading dataset...
Total images: 1500
Total annotations: 4784
Original categories: 60
New categories: 9
Updating annotations...
Saving merged dataset...
Done!
Saved to: C:\Users\User\Downloads\archive\data\annotations_merged.json


In [36]:
from collections import Counter

counts = Counter()

for ann in annotations:
    counts[ann["category_id"]] += 1

print("\nNew class distribution:\n")

for cat in new_categories:
    cid = cat["id"]
    name = cat["name"]
    print(name, ":", counts[cid])


New class distribution:

Bottle : 285
Cigarette : 667
Foam : 140
Glass : 254
Metal : 550
Other : 108
Paper : 497
Plastic : 1766
Unlabeled : 517


In [38]:
import json
import os
import cv2

# =========================
# PATHS
# =========================

IMAGE_ROOT = r"C:\Users\User\Downloads\archive\data"
ANNOTATION_FILE = r"C:\Users\User\Downloads\archive\data\annotations_merged.json"

OUTPUT_IMAGE_DIR = r"C:\Users\User\Downloads\archive\processed_images"
OUTPUT_JSON = r"C:\Users\User\Downloads\archive\annotations_resized.json"

TARGET_SIZE = 1024

os.makedirs(OUTPUT_IMAGE_DIR, exist_ok=True)

# =========================
# LOAD DATA
# =========================

with open(ANNOTATION_FILE) as f:
    data = json.load(f)

images = data["images"]
annotations = data["annotations"]

# create annotation lookup
ann_by_image = {}

for ann in annotations:
    ann_by_image.setdefault(ann["image_id"], []).append(ann)

# =========================
# PROCESS IMAGES
# =========================

for img in images:

    file_name = img["file_name"]
    image_id = img["id"]

    img_path = os.path.join(IMAGE_ROOT, file_name)

    image = cv2.imread(img_path)

    if image is None:
        print("Skipped:", file_name)
        continue

    h, w = image.shape[:2]

    # resize
    resized = cv2.resize(image, (TARGET_SIZE, TARGET_SIZE))

    # scaling factors
    scale_x = TARGET_SIZE / w
    scale_y = TARGET_SIZE / h

    # update bounding boxes
    for ann in ann_by_image.get(image_id, []):

        x, y, bw, bh = ann["bbox"]

        ann["bbox"] = [
            x * scale_x,
            y * scale_y,
            bw * scale_x,
            bh * scale_y
        ]

    # save resized image
    save_path = os.path.join(OUTPUT_IMAGE_DIR, file_name.replace("/", "_"))

    cv2.imwrite(save_path, resized)

    # update image size
    img["width"] = TARGET_SIZE
    img["height"] = TARGET_SIZE

print("Images processed.")

# =========================
# SAVE NEW ANNOTATION FILE
# =========================

with open(OUTPUT_JSON, "w") as f:
    json.dump(data, f)

print("Resized dataset saved.")

Images processed.
Resized dataset saved.


NameError: name 'category_lookup' is not defined